# **ENVIROMENT INITIALIZATION**

In [1]:
# IMPORTS
# Math libraries
import numpy as np
from scipy import stats
import pandas as pd
import polars as pl

# Plots
import matplotlib.pyplot as plt

# Technical libraries
from pathlib import Path
import os
from datetime import date, timedelta

In [2]:
# CONFIGURATION
from config import (
    PROJECT_ROOT,
    AS_OF, TARGET_START, TARGET_END,
    DATA_RAW_DIR, DATA_PROCESSED_DIR, DATA_FEATURES_DIR,
    TRAIN_PATH, CV_TARGET_PATH, HISTORY_PATH, CV_FEATURES_PATH,
    REPORTS_DIR, REPORTS_GMV_DIR, REPORTS_CONV_FUNL_DIR, REPORTS_FEATURES_DIR,
    WINDOWS,
    GMV_COLS, ACTIVITY_COLS, CONVERSION_COLS,
    BIN_ORDER,
    create_directories,
    validate_data_exists,
)

# Creating directories
create_directories()

# Checking for data availability
validate_data_exists()

# Configuration info
print("DATA TRANSFORMING CONFIGURATION")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"TRAIN_PATH: {TRAIN_PATH}")
print(f"TARGET_PATH: {CV_TARGET_PATH}")
print(f"CV_FEATURES_PATH: {CV_FEATURES_PATH}")
print(f"AS_OF: {AS_OF}")
print(f"TARGET_START: {TARGET_START}")
print(f"TARGET_END: {TARGET_END}")
print(f"WINDOWS: {WINDOWS}")

All directories created successfully.
TRAIN_PATH: 171.80 MB
CV_TARGET_PATH: 1.32 MB

All data files are present.
DATA TRANSFORMING CONFIGURATION
PROJECT_ROOT: D:\.workspace\Programming\Projects\E-cup_2026_by_Ozon_Tech
TRAIN_PATH: D:\.workspace\Programming\Projects\E-cup_2026_by_Ozon_Tech\data\raw\train.parquet
TARGET_PATH: D:\.workspace\Programming\Projects\E-cup_2026_by_Ozon_Tech\data\processed\cv_target_2026-01-14.parquet
CV_FEATURES_PATH: D:\.workspace\Programming\Projects\E-cup_2026_by_Ozon_Tech\data\processed\features\cv_features_2026-01-14.parquet
AS_OF: 2026-01-14
TARGET_START: 2026-01-15
TARGET_END: 2026-02-13
WINDOWS: [7, 14, 30, 60, 90, 180, 365]


## **CUSTOM FUNCTIONS**

In [3]:
# FUNCTION FOR CONVERTING POLAR TABLES IN EDA
def print_vertical_one_row(df: pl.DataFrame) -> None:
    """
    It conveniently prints wide tables with a single line.
    If there are more lines, it prints a table with expanded output.
    """
    if df.height == 0:
        print("Empty DataFrame")
        return

    if df.height == 1:
        row = df.row(0)
        for col_name, value in zip(df.columns, row):
            print(f"{col_name:45} {value}")
    else:
        with pl.Config() as cfg:
            cfg.set_tbl_cols(-1)
            cfg.set_tbl_width_chars(-1)
            cfg.set_tbl_rows(-1)
            print(df)

# **DATA REBUILDING**

In [4]:
# DATA LOADING
train_lf = pl.scan_parquet(TRAIN_PATH)

# Sanity check
train_info = train_lf.select(
    pl.col("event_date").min().alias("min_date"),
    pl.col("event_date").max().alias("max_date"),
    pl.len().alias("rows"),
    pl.col("user_id").n_unique().alias("users"),
).collect()

print(f"Train data loaded: {train_lf.collect_schema()}")
print(train_info)

Train data loaded: Schema({'event_date': Date, 'user_id': Int64, 'search': Int64, 'cat': Int64, 'has_search_to_cart': Int64, 'has_search_to_ord': Int64, 'has_cat_to_cart': Int64, 'has_cat_to_ord': Int64, 'search_to_cart': Int64, 'search_to_ord': Int64, 'cat_to_cart': Int64, 'cat_to_ord': Int64, 'gmv_search': Float64, 'gmv_cat': Float64, 'to_cart': Int64, 'to_ord': Int64, 'gmv': Float64, 'searches': Int64})
shape: (1, 4)
┌────────────┬────────────┬──────────┬────────┐
│ min_date   ┆ max_date   ┆ rows     ┆ users  │
│ ---        ┆ ---        ┆ ---      ┆ ---    │
│ date       ┆ date       ┆ u32      ┆ u32    │
╞════════════╪════════════╪══════════╪════════╡
│ 2025-01-01 ┆ 2026-02-13 ┆ 30631006 ┆ 250000 │
└────────────┴────────────┴──────────┴────────┘


In [5]:
# CREATE HISTORY SNAPSHOT
# Filtering only the data up to AS_OF
history_lf = train_lf.filter(pl.col("event_date") <= AS_OF)

# Sanity check
history_info = history_lf.select(
    pl.col("event_date").min().alias("min_date"),
    pl.col("event_date").max().alias("max_date"),
    pl.len().alias("rows"),
    pl.col("user_id").n_unique().alias("users"),
).collect()

print("History snapshot info:")
print(history_info)

history_lf.sink_parquet(HISTORY_PATH)
print(f"\nHistory saved to: {HISTORY_PATH}")
print(f"File size: {HISTORY_PATH.stat().st_size / 1024 / 1024:.2f} MB")

History snapshot info:
shape: (1, 4)
┌────────────┬────────────┬──────────┬────────┐
│ min_date   ┆ max_date   ┆ rows     ┆ users  │
│ ---        ┆ ---        ┆ ---      ┆ ---    │
│ date       ┆ date       ┆ u32      ┆ u32    │
╞════════════╪════════════╪══════════╪════════╡
│ 2025-01-01 ┆ 2026-01-14 ┆ 27837039 ┆ 250000 │
└────────────┴────────────┴──────────┴────────┘

History saved to: D:\.workspace\Programming\Projects\E-cup_2026_by_Ozon_Tech\data\processed\history_before_2026-01-14.parquet
File size: 158.26 MB


In [6]:
# CREATE TARGET
# Filtering only the data after AS_OF
cv_target_lf = train_lf.filter(
    (pl.col("event_date") > AS_OF) &
    (pl.col("event_date") <= TARGET_END)
)

# Sanity check
future_info = cv_target_lf.select(
    pl.col("event_date").min().alias("min_target_date"),
    pl.col("event_date").max().alias("max_target_date"),
    pl.len().alias("rows_in_target_window"),
    pl.col("user_id").n_unique().alias("users_in_target_window"),
).collect()

print("Target window info:")
print(future_info)

# Aggregating GMV by users
target_lf = (
    cv_target_lf
    .group_by("user_id")
    .agg(pl.col("gmv").sum().alias("target"))
)

# Getting a list of all users
users_lf = train_lf.select(pl.col("user_id").unique())

# Join with zero filling for users without purchases
cv_target_lf = (
    users_lf
    .join(target_lf, on = "user_id", how = "left")
    .with_columns(pl.col("target").fill_null(0.0))
)

cv_target = cv_target_lf.collect()

# Target statistics
cv_target_stats = cv_target_lf.select(
    pl.len().alias("users"),
    pl.col("target").mean().alias("mean"),
    pl.col("target").median().alias("median"),
    pl.col("target").max().alias("max"),
    (pl.col("target") == 0).mean().alias("zero_share"),
    (pl.col("target") > 0).mean().alias("positive_share"),
    pl.col("target").quantile(0.50).alias("q50"),
    pl.col("target").quantile(0.90).alias("q90"),
    pl.col("target").quantile(0.99).alias("q99"),
    pl.col("target").quantile(0.999).alias("q999"),
).collect()

print("\nTarget statistics:")
print(cv_target_stats)

cv_target.write_parquet(CV_TARGET_PATH)
print(f"\nTarget saved to: {CV_TARGET_PATH}")

Target window info:
shape: (1, 4)
┌─────────────────┬─────────────────┬───────────────────────┬────────────────────────┐
│ min_target_date ┆ max_target_date ┆ rows_in_target_window ┆ users_in_target_window │
│ ---             ┆ ---             ┆ ---                   ┆ ---                    │
│ date            ┆ date            ┆ u32                   ┆ u32                    │
╞═════════════════╪═════════════════╪═══════════════════════╪════════════════════════╡
│ 2026-01-15      ┆ 2026-02-13      ┆ 2793967               ┆ 250000                 │
└─────────────────┴─────────────────┴───────────────────────┴────────────────────────┘

Target statistics:
shape: (1, 10)
┌────────┬───────────┬──────────┬────────────┬───┬──────────┬────────────┬────────────┬────────────┐
│ users  ┆ mean      ┆ median   ┆ max        ┆ … ┆ q50      ┆ q90        ┆ q99        ┆ q999       │
│ ---    ┆ ---       ┆ ---      ┆ ---        ┆   ┆ ---      ┆ ---        ┆ ---        ┆ ---        │
│ u32    ┆ f64     

# **FEATURE ENGINEERING**

| Категория | Фичи | Обоснование |
|---|---|---|
| **Recency** | days_since_last_activity, days_since_last_purchase, days_since_last_order, days_since_last_cart, days_since_last_search | Пользователи с недавней активностью конвертируют лучше |
| **Frequency** | active_days_last_7/14/30/60/90d, purchase_days_last_7/14/30/60/90d, orders_last_7/14/30/60/90d, searches_last_7/14/30/60/90d | Частота — главный драйвер различий (328 vs 7 заказов) |
| **Monetary** | gmv_last_7/14/30/60/90/180/365d, avg_order_value_last_30d, avg_order_value_last_90d | GMV за разные окна, средний чек |
| **Conversion** | cart_to_order_rate, search_to_cart_rate, search_to_order_rate, cat_to_cart_rate, cat_to_order_rate | Конверсии на разных этапах воронки |
| **Channel** | search_share, cat_share, uses_both_channels_hist, gmv_search_share | Доля каналов в активности |
| **Trend** | gmv_7d / gmv_30d, gmv_30d / gmv_90d, active_days_7d / active_days_30d | Ускорение/замедление активности |
| **Seasonal** | weekday, is_weekend, week_of_month, days_to_next_salary, days_since_last_salary, days_to_holiday, is_holiday_week | Зарплатные циклы, праздники, недельная сезонность |
| **Funnel** | avg_items_per_order, avg_cart_adds_per_day, avg_searches_per_day | Интенсивность на каждом этапе |
| **Historical** | gmv_same_period_last_year (если данные позволяют) | YoY baseline |